# **PISA STUDENTS**

## Imports

In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from models import train_linear_model, train_logistic_model
from sklearn.metrics import accuracy_score
from data import category_split, balance_feature, frame_status_info
from evaluation import evaluate_model, plot_model_acc_and_fscore, display_model_weights, evaluate_model_thresholded_regression, evaluate_model_classifier

Using device: cpu


## Settings

In [2]:
DATA_RELATIVE_PATH = "data/pisa_2022/pisa_clean.csv"

TARGET_COLUMN = "GENERAL_SCORE"
TARGET_OPERTOR = "<" #Does not change the training, only splits
TARGET_VALUE = 400 #This value will be changed in a future cell
PERCENTILE = 10


RANDOM_SEED = None
SPLIT_PERCENTAGE_TESTING = 0.15

BALANCE_DATA = False
BALANCE_QUERY = TARGET_COLUMN + " " + TARGET_OPERTOR + " " + str(TARGET_VALUE) #This value is updated in future cell
BALANCE_QUERY_NON_TARGET_PERCENTAGE = 0.50

MODEL_BALANCE = True
BALANCE = "balanced" if MODEL_BALANCE else None


REGULARIZER_WEIGTH = 0.1
#MODEL_THERESHOLD_REG = TARGET_VALUE
MODEL_THERESHOLD_LOG = 0.5

DEBUG_MODE = True


## Loading the Data

In [3]:
# Load dataset
data_file = os.path.join(os.getcwd(), DATA_RELATIVE_PATH)
df = pd.read_csv(data_file)
print("Loaded datafile")
print(df.shape)

Loaded datafile
(6072, 521)


# Data processing

### Irrelevant data

In [4]:
#Remove constant columns
df = df.loc[:, df.nunique() > 1]
print(df.shape)

(6072, 520)


### Identical Data

In [5]:
# Ensure all rows are unique
df = df.drop_duplicates()
print("Removed duplicate rows")
print(df.shape)

Removed duplicate rows
(6072, 520)


### Missing data

In [6]:
# Investigation
# Count missing values per column
missing_counts = df.isnull().sum()

# Total number of rows
total_rows = len(df)

# Create summary DataFrame
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percent': (missing_counts / total_rows * 100).round(2)
})

# Filter to only show columns with missing data
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]

# Sort by most missing values
missing_summary = missing_summary.sort_values(by='Missing Percent', ascending=False)

# Display result
print(missing_summary.head(120))  # Show top 20, adjust as needed

Empty DataFrame
Columns: [Missing Count, Missing Percent]
Index: []


In [7]:
# All data is present in this dataframe.
# Fill NaN values in numeric columns with the column mean
df = df.apply(lambda col: col.fillna(col.mean()) if pd.api.types.is_numeric_dtype(col) else col)
print(df.shape)

(6072, 520)


### Data transformation

In [8]:
# Round all float columns to 2 decimal places
df = df.apply(lambda col: col.round(2) if pd.api.types.is_float_dtype(col) else col)
print(df.shape)

(6072, 520)


### POST REMOVAL

In [9]:
print(df.shape)

(6072, 520)


## Feature Processing

### Columns to be removed

In [10]:
#Remove columns
#columns_to_remove = ["CNTSTUID", "ST004D01T", "AGE", "SCIENCE_AVG", "MATH_AVG", "READING_AVG", "ST251D08JA", "OCOD1", "OCOD2", "OCOD3", "PROGN", "ISCEDP", "COBN_S", "COBN_M", "COBN_F", "LANGN"]
#df.drop(columns=columns_to_remove, inplace=True, errors='ignore')
print(df.shape)

(6072, 520)


### Prediction column analysis

In [11]:
# Calculate the score corresponding to that percentile
THRESHOLD_SCORE = df['GENERAL_SCORE'].quantile(PERCENTILE / 100)

print(f"{PERCENTILE}th percentile score: {THRESHOLD_SCORE:.2f}")

# Optionally: Filter students at or below that percentile
lowest_students = df[df['GENERAL_SCORE'] <= THRESHOLD_SCORE]

print(f"Number of students at or below the {PERCENTILE}th percentile: {len(lowest_students)}")
print(lowest_students[['GENERAL_SCORE']].head())

TARGET_VALUE = THRESHOLD_SCORE
BALANCE_QUERY = TARGET_COLUMN + " " + TARGET_OPERTOR + " " + str(TARGET_VALUE)

10th percentile score: 357.31
Number of students at or below the 10th percentile: 608
    GENERAL_SCORE
25         351.27
41         324.81
44         356.80
46         341.33
60         325.31


## Splitting data

### Split

In [12]:
train_df, test_df = train_test_split(df, test_size=SPLIT_PERCENTAGE_TESTING, random_state=RANDOM_SEED)
frame_status_info(train_df, test_df)

Training data shape: (5161, 520)
Testing data shape: (911, 520)


### Balance

In [13]:
if (BALANCE_DATA):
    match_df, no_match_df = category_split(train_df, BALANCE_QUERY)
    train_balanced = balance_feature(match_df, no_match_df, BALANCE_QUERY_NON_TARGET_PERCENTAGE)

## Training

### Preparation

In [14]:
# Define features and target variable
temp_train : pd.DataFrame = train_balanced if BALANCE_DATA else train_df

frame_status_info(temp_train, test_df)
print("Training percentage:", test_df.shape[0]/(temp_train.shape[0]+test_df.shape[0]))

X_train = temp_train.drop(columns=[TARGET_COLUMN]).values
y_train_reg = temp_train[TARGET_COLUMN]
y_train_cls = (temp_train[TARGET_COLUMN] < TARGET_VALUE).astype(int)

X_test = test_df.drop(columns=[TARGET_COLUMN]).values
y_test_reg = test_df[TARGET_COLUMN]
y_test_cls = (test_df[TARGET_COLUMN] < TARGET_VALUE).astype(int)

Training data shape: (5161, 520)
Testing data shape: (911, 520)
Training percentage: 0.15003293807641635


### Execution

In [ ]:
# Train regression models
print("Training RegN")
model_reg_no = train_linear_model(X_train, y_train_reg.values, reg_type=None, class_weight=BALANCE)  # Standard
print("Training Reg1")
model_reg_l1 = train_linear_model(X_train, y_train_reg.values, reg_type="l1", alpha=REGULARIZER_WEIGTH, class_weight=BALANCE)  # Lasso
print("Training Reg2")
model_reg_l2 = train_linear_model(X_train, y_train_reg.values, reg_type="l2", alpha=REGULARIZER_WEIGTH, class_weight=BALANCE)  # Ridge

# Train logistic models
print("Training LogN")
model_log_no = train_logistic_model(X_train, y_train_cls.values, reg_type=None, class_weight=BALANCE)  # Standard
print("Training Log1")
model_log_l1 = train_logistic_model(X_train, y_train_cls.values, reg_type="l1", alpha=REGULARIZER_WEIGTH, class_weight=BALANCE)  # Lasso
print("Training Log2")
model_log_l2 = train_logistic_model(X_train, y_train_cls.values, reg_type="l2", alpha=REGULARIZER_WEIGTH, class_weight=BALANCE)  # Ridge

Training RegN


TypeError: SGDRegressor.__init__() got an unexpected keyword argument 'class_weight'

## Evaluation

### Execution

In [ ]:
# Evaluate regression models
print("Regression Models:")
acc_reg_no, f_reg_no = evaluate_model_thresholded_regression(model_reg_no, X_test, (y_test_reg < TARGET_VALUE).astype(int), TARGET_VALUE, "Standard Linear Regression", debug=DEBUG_MODE)
acc_reg_l1, f_reg_l1 = evaluate_model_thresholded_regression(model_reg_l1, X_test, (y_test_reg < TARGET_VALUE).astype(int), TARGET_VALUE, "Lasso Regression", debug=DEBUG_MODE)
acc_reg_l2, f_reg_l2 = evaluate_model_thresholded_regression(model_reg_l2, X_test, (y_test_reg < TARGET_VALUE).astype(int), TARGET_VALUE, "Ridge Regression", debug=DEBUG_MODE)

# Evaluate logistic models
print("Logistic Models:")
acc_log_no, f_log_no = evaluate_model_classifier(model_log_no, X_test, y_test_cls, MODEL_THERESHOLD_LOG, "Standard Logistic Regression", debug=DEBUG_MODE)
acc_log_l1, f_log_l1 = evaluate_model_classifier(model_log_l1, X_test, y_test_cls, MODEL_THERESHOLD_LOG, "Lasso Logistic Regression", debug=DEBUG_MODE)
acc_log_l2, f_log_l2 = evaluate_model_classifier(model_log_l2, X_test, y_test_cls, MODEL_THERESHOLD_LOG, "Ridge Logistic Regression", debug=DEBUG_MODE)

### Analysis

In [ ]:
accs = [acc_reg_no, acc_reg_l1, acc_reg_l2, acc_log_no, acc_log_l1, acc_log_l2]
fscores = [f_reg_no, f_reg_l1, f_reg_l2, f_log_no, f_log_l1, f_log_l2]
model_names = ["Linear Regression", "Lasso Regression", "Ridge Regression",
               "Logistic Regression", "Lasso LogRegression", "Ridge LogRegression"]
plot_model_acc_and_fscore(accs, fscores, model_names)

### Weights

In [ ]:
display_model_weights(model_reg_no, train_df.drop(columns=[TARGET_COLUMN]), num_features=10)

In [ ]:
display_model_weights(model_reg_l1, train_df.drop(columns=[TARGET_COLUMN]), num_features=10)

In [ ]:
display_model_weights(model_reg_l2, train_df.drop(columns=[TARGET_COLUMN]), num_features=10)

In [ ]:
display_model_weights(model_reg_no, train_df.drop(columns=[TARGET_COLUMN]), num_features=10)

In [ ]:
display_model_weights(model_log_l1, train_df.drop(columns=[TARGET_COLUMN]), num_features=10)

In [ ]:
display_model_weights(model_log_l2, train_df.drop(columns=[TARGET_COLUMN]), num_features=10)